# Lab 3 — Swap in a real model backend safely

The Six Hats architecture is provider-independent. Every agent only needs a backend with one method:

```python
complete(system_prompt: str, user_prompt: str) -> str
```

This notebook shows the adapter pattern without putting a secret in the browser.

In [ ]:
from six_hats_agents import CallableBackend, SixHatsOrchestrator, TeachingBackend

## A callable adapter

The `CallableBackend` wraps any function that takes a system prompt and user prompt and returns text.

In [ ]:
def my_safe_model_call(system_prompt: str, user_prompt: str) -> str:
    # Replace this function only when you have a secure server-side endpoint.
    # Do NOT paste a long-lived provider API key into a public JupyterLite notebook.
    return TeachingBackend().complete(system_prompt, user_prompt)

backend = CallableBackend(my_safe_model_call)
orchestrator = SixHatsOrchestrator(backend=backend)
result = orchestrator.run("How should we evaluate a new decision-support tool before broad adoption?")
print(result["synthesis"])

## Recommended production boundary

```text
JupyterLite notebook in browser
          |
          | HTTPS request authenticated for your application
          v
Small server / API gateway
          |
          | provider secret stays here
          v
LLM provider
```

Your server endpoint should enforce authentication, authorization, rate limits, input/output size limits, logging policy, and any organizational data-handling requirements. It should return only the model result the notebook needs.

For sensitive or high-impact use cases, add human review and keep an audit trail of the original problem, evidence, agent outputs, synthesis, and final decision.

## Build a real synthesis prompt

The module also includes `build_llm_synthesis_prompt(...)`. It bundles the six outputs for a seventh model call while telling the Orchestrator not to erase disagreement or equate consensus with truth.

In [ ]:
from six_hats_agents import build_llm_synthesis_prompt

system_prompt, user_prompt = build_llm_synthesis_prompt(
    result["problem"], result["perspectives"]
)
print(system_prompt)
print("\n--- USER PROMPT PREVIEW ---\n")
print(user_prompt[:1800])


## Extension ideas

- Require JSON outputs from each hat.
- Add citations to White Hat evidence.
- Give agents different tool permissions instead of giving every tool to every agent.
- Record confidence and unresolved questions.
- Add a critic/revision pass.
- Compare the six-agent workflow against a single-agent baseline on a fixed evaluation set.